# Jev-Omni, measured three ways: re-analysis from the saved answers

Companion to [the post on condados.ai](https://condados.ai/blog/jev-omni-decision-head-benchmark).
No GPU needed. It reads the per-question answers in `output/results/`, recomputes the post's tables
with the functions in `src/analyze.py`, and checks the whole result against `output/analysis.json`.

Readouts: **A** `jev` = Jev-Omni through its decision head · **B** `base` = untouched Gemma 4 12B IT
through the probability of writing each option number · **C** `jevdigits` = Jev-Omni's fine-tuned
weights read the same way as B.

In [ ]:
# Python 3.10+ standard library only; nothing to install.
import json, os, subprocess, sys
from pathlib import Path

if not Path("output/results").exists() and Path("../output/results").exists():
    os.chdir("..")  # opened from notebooks/
if not Path("output/results").exists():  # on Colab: fetch the repository
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/CondadosAI/jev-omni-eval"], check=True)
    os.chdir("jev-omni-eval")
sys.path.insert(0, "src")
import analyze

data = {}
for f in sorted(Path("output/results").glob("*__*.json")):
    bench, arm = f.stem.split("__")
    data[(bench, arm)] = {analyze.key(r): r for r in json.loads(f.read_text())}
print(len(data), "result files")

## Accuracy: each benchmark's headline metric, and per question

Headline ("native"): mean over case logs on DecisionBench, mean over the 14 tasks on BLINK,
per question on MMStar.

In [ ]:
for bench in ("medium", "hard", "blink", "mmstar"):
    row = []
    for L, arm in (("A", "jev"), ("B", "base"), ("C", "jevdigits")):
        micro, native = analyze.accuracy(bench, list(data[(bench, arm)].values()))
        row.append(f"{L} {native:6.2f} ({micro:6.2f} per question)")
    print(f"{bench:7s}", " | ".join(row))

## Paired differences against B

McNemar's exact test on the questions where the two readouts disagree, and 95% bootstrap intervals
that resample case logs on DecisionBench and questions within tasks on BLINK.

In [ ]:
for other, L in (("jev", "A"), ("jevdigits", "C")):
    for bench in ("medium", "hard", "blink", "mmstar"):
        p = analyze.paired(bench, data[(bench, other)], data[(bench, "base")])
        lo, hi = p["micro_diff_ci95_pp"]
        print(f"{L} - B  {bench:7s} {p['micro_diff_pp']:+6.2f} [{lo:+.2f}, {hi:+.2f}]   "
              f"disagreements {p['a_only']} vs {p['b_only']}, McNemar p = {p['mcnemar_p']}")

## Calibration: ECE before and after one temperature per readout

T is fitted by negative log-likelihood on DecisionBench medium and applied unchanged elsewhere.

In [ ]:
T = {arm: analyze.fit_T(list(data[("medium", arm)].values())) for arm in ("jev", "base", "jevdigits")}
print("T:", T)
for bench in ("medium", "hard", "blink", "mmstar"):
    cells = []
    for L, arm in (("A", "jev"), ("B", "base"), ("C", "jevdigits")):
        rows = list(data[(bench, arm)].values())
        cells.append(f"{L} {analyze.stats(rows)['ece']:.3f} -> {analyze.stats(rows, T[arm])['ece']:.3f}")
    print(f"{bench:7s}", " | ".join(cells))

## Readout C leans on option 1 when it reads an image

In [ ]:
for bench in ("blink", "mmstar"):
    for L, arm in (("B", "base"), ("C", "jevdigits")):
        rows = list(data[(bench, arm)].values())
        pb = analyze.position_bias(rows)
        print(bench, L, pb["by_n_options"], "prior-corrected accuracy:", pb["acc_micro_prior_corrected"])

## Check against `output/analysis.json`

Recompute the full analysis with `src/analyze.py` and compare it to the committed file.

In [ ]:
out = subprocess.run([sys.executable, "src/analyze.py"], capture_output=True, text=True, check=True).stdout
assert json.loads(out) == json.loads(Path("output/analysis.json").read_text())
print("output/analysis.json reproduces exactly")